# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 14 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficiencies page',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-a

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn company page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Forum / community page', 'url': 'https://discuss.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
moonshotai/Kimi-K2.5
Updated
3 days ago
•
96.2k
•
1.51k
Tongyi-MAI/Z-Image
Updated
6 days ago
•
6.35k
•
809
tencent/HunyuanImage-3.0-Instruct
Updated
6 days ago
•
148
•
784
deepseek-ai/DeepSeek-OCR-2
Updated
about 3 hours ago
•
144k
•
644
nvidia/personaplex-7b-v1
Updated
5 days ago
•
101k
•
1.6k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.15k
Qwen3-TTS Demo
🎙
1.15k
Generate realistic speech from text with custom voices or voice cloning
Running
on
Zero
MCP
1.96k
Z Image Turbo
🖼
1.96k
Generate stunning images from text descript

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmoonshotai/Kimi-K2.5\nUpdated\n3 days ago\n•\n96.2k\n•\n1.51k\nTongyi-MAI/Z-Image\nUpdated\n6 days ago\n•\n6.35k\n•\n809\ntencent/HunyuanImage-3.0-Instruct\nUpdated\n6 days ago\n•\n148\n•\n784\ndeepseek-ai/DeepSeek-OCR-2\nUpdated\nabout 3 hours ago\n•\n144k\n•\n644\nnvidia/personaplex-7b-v1\nUpdated\n5 days ago\n•\n101k\n•\n1.6k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.15k

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a leading platform and vibrant community dedicated to advancing machine learning (ML) through open collaboration. Positioned as **The Home of Machine Learning**, Hugging Face empowers engineers, scientists, and AI enthusiasts worldwide to create, discover, and share cutting-edge ML models, datasets, and applications.

The platform hosts **over 2 million ML models**, **500,000+ datasets**, and **1 million+ applications**, supporting all modalities including text, image, video, audio, and 3D data. It is the go-to destination for anyone looking to innovate and accelerate AI development using a rich ecosystem of open-source tools and resources.

---

## What Hugging Face Offers

- **Hugging Face Hub**: A central space for sharing and exploring open-source ML models and datasets.
- **Spaces**: Hosting and deploying AI-powered applications in a collaborative environment.
- **Open-source Stack**: Tools and libraries to build, test, and scale AI products.
- **Compute Access**: Paid compute resources to accelerate machine learning workflows.
- **Multi-Modality Support**: Work with text, images, videos, audio, and 3D easily.
- **Community Resources**: Tutorials, documentation, and discussions to foster learning and collaboration.

Trending models on the platform feature applications such as realistic text-to-speech synthesis, video generation from images, and advanced image manipulation.

---

## Company Culture

Hugging Face thrives on openness, collaboration, and ethics. It fosters a global community where knowledge sharing and innovation come together to build an open and ethical AI future. The company encourages its community members to build portfolios, contribute to open-source projects, and actively participate in advancing AI technologies.

The environment is inclusive and mission-driven, attracting passionate individuals who want to shape machine learning’s future by working on impactful projects alongside leading ML experts and organizations.

---

## Customers & Community

Hugging Face serves a broad customer base including:

- **Individual ML engineers and researchers** seeking state-of-the-art models and datasets.
- **AI enthusiasts and developers** building applications across modalities.
- **Enterprises** looking to integrate powerful ML solutions and accelerate AI-driven innovation.
- **Academic and research institutions** advancing AI through open collaboration.
  
The community is one of the fastest growing in AI, with thousands of active contributors constantly updating and improving resources on the platform.

---

## Careers & Opportunities

Joining Hugging Face means becoming part of a pioneering and collaborative AI community. Career opportunities often span roles in:

- Machine Learning Engineering
- Research Science
- Software Development
- DevOps and Cloud Infrastructure
- Community Engagement and Developer Relations

The company values open-source contribution experience, a strong commitment to ethical AI, and enthusiasm for collaboration. Employees enjoy working in an innovative, transparent environment that facilitates professional growth and impact.

---

## Get Involved

- **Explore AI Apps:** Try and showcase applications across different AI modalities.
- **Browse Hugging Face Hub:** Access millions of models and datasets.
- **Share Your Work:** Build your public ML portfolio and collaborate with the community.
- **Access Compute Resources:** Accelerate your machine learning projects with paid compute services.

Sign up today and join the AI community that’s shaping tomorrow’s technology!

---

## Contact & More Information

Website: [huggingface.co](https://huggingface.co)

---

**Hugging Face** — Building an open, ethical, and collaborative future for artificial intelligence.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It serves as a vibrant collaboration platform where machine learning engineers, scientists, and enthusiasts come together to create, discover, and share models, datasets, and applications. With over 2 million models and 500,000+ datasets accessible, Hugging Face is the home of open and ethical AI development.

---

## What We Offer

- **Models:** Access and collaborate on a vast collection of cutting-edge machine learning models for text, image, video, audio, 3D, and more.
- **Datasets:** Find and contribute to hundreds of thousands of datasets to advance AI research.
- **Spaces:** Deploy and experiment with AI applications in an easy-to-use space to showcase your projects or explore others’ works.
- **Community:** Join a fast-growing community dedicated to democratizing AI with open source tools and ethical standards.
- **Enterprise:** Customized solutions for businesses to accelerate machine learning efforts.
- **Docs and Learning:** Comprehensive documentation and learning tracks like *Hugging Face Fundamentals* provide foundational skills.

---

## Our Mission & Values

- **Democratizing machine learning:** Making cutting-edge AI accessible to everyone.
- **Open collaboration:** Encouraging a community-driven approach that nurtures innovation through sharing and collective growth.
- **Ethical AI:** Committed to building AI responsibly through transparency and openness.
- **Supporting all modalities:** From text and speech to 3D imagery, Hugging Face covers all the latest AI modalities.
- **Empowering creators:** Helping engineers and researchers build their portfolio and showcase their work worldwide.

---

## Our Community

- Over 80,000+ active followers deeply invested in AI & ML.
- Continuous activity from contributors sharing papers, datasets, and models daily.
- Supportive environment open to new members who want to contribute to groundbreaking AI solutions.
- Events, articles, and papers published regularly fostering knowledge exchange and visibility.

---

## Careers at Hugging Face

Are you passionate about transforming AI? Hugging Face invites ambitious individuals who want to make machine learning open, accessible, and ethical to join our team. Here, you will:

- Collaborate with world-class AI researchers and engineers.
- Work on state-of-the-art machine learning tools and libraries used globally.
- Contribute to an open-source ecosystem with real-world impact.
- Thrive in a culture valuing innovation, openness, and community-driven success.

*Join us on our mission to democratize machine learning — one commit at a time.*

---

## Contact & Further Information

- Website: [huggingface.co](https://huggingface.co)
- Press inquiries: Contact via the Hugging Face team page
- Explore models, datasets, and applications at Hugging Face Hub

---

## Brand Essence

- **Logo Colors:** Bright, friendly yellow (#FFD21E, #FF9D00) paired with neutral gray (#6B7280)
- **Tagline:** *The AI community building the future.*
- **Style:** Open, collaborative, innovative, and community-focused.

---

Unlock the power of machine learning with Hugging Face — where AI is built together.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>